### Kitti Coordinates Convention
<img src="./kitti_convention.png" width=400 height=400 />

In [ ]:
import os 
import sys
import numpy as np
import matplotlib.pyplot as plt

import struct
from open3d import * 
import open3d as o3d
import copy
import math
import random
from operator import itemgetter

from utils_3d import *
from detection_v2_time import *
from discretize import TensorMesh

from joblib import dump, load
from sklearn.cluster import DBSCAN
from sklearn import metrics
from sklearn.preprocessing import StandardScaler

In [ ]:
def get_grd_lvl_point_cloud(lidar_file, height = -1.73 , delta=0.2, display = False) :
    lidar_array = np.fromfile(lidar_file, dtype =np.float32).reshape(-1,4)
    lidar_array = np.delete(lidar_array,3,1)

    pcd = o3d.geometry.PointCloud()
    pcd.points = o3d.utility.Vector3dVector(lidar_array.reshape(-1,3))

    ## get all points just above ground level (for V1 - its BEV)
    res_tmp = np.where(lidar_array >= height-delta) # -1.73 is ground
    res = np.where(lidar_array <= height+delta)
    interest = np.where(res[1] == 2)
    points = res[0][interest]
    grd = [[0,0,0]]
    grd = np.reshape(grd,(-1,3))
    for index in points :
        coords = lidar_array[index]
        coords = np.reshape(coords,(-1,3))
        grd = np.append(grd, coords, axis = 0)

    ground_pcd = o3d.geometry.PointCloud()
    ground_pcd.points = o3d.utility.Vector3dVector(grd)
    if display :
        custom_draw_geometry_with_custom_fov([ground_pcd], 90.0)
    return ground_pcd

In [ ]:
class Cell(object) :
    def __init__ (self, center_, width=0.6) :
        half_width = width/2
        self.center = center_ # x,y,z
        self.corners = [
            [self.center[0]-half_width, self.center[1]-half_width, self.center[2]-half_width], # bottom left corner
            [self.center[0]+half_width, self.center[1]-half_width, self.center[2]-half_width], # bottom right corner
            [self.center[0]+half_width, self.center[1]+half_width, self.center[2]-half_width], # top right corner
            [self.center[0]-half_width, self.center[1]+half_width, self.center[2]-half_width], # top left corner
            [self.center[0]-half_width, self.center[1]-half_width, self.center[2]+half_width], # bottom left corner
            [self.center[0]+half_width, self.center[1]-half_width, self.center[2]+half_width], # bottom right corner
            [self.center[0]+half_width, self.center[1]+half_width, self.center[2]+half_width], # top right corner
            [self.center[0]-half_width, self.center[1]+half_width, self.center[2]+half_width], # top left corner
        ]
        self.occupied = False
    
    def set_free(self):
        self.occupied = False
        return
    def set_occupied(self):
        self.occupied = True
        return
    def is_occupied(self) :
        return self.occupied
    def print_corners(self) :
        print(self.corners)
    def point_in_cell(self, pt) :
        # generate faces
        f1 =  Face([Vector(self.corners[0]), Vector(self.corners[1]), Vector(self.corners[5]), Vector(self.corners[4])])
        f2 =  Face([Vector(self.corners[3]), Vector(self.corners[7]), Vector(self.corners[6]), Vector(self.corners[2])])
        f3 =  Face([Vector(self.corners[0]), Vector(self.corners[4]), Vector(self.corners[7]), Vector(self.corners[3])])
        f4 =  Face([Vector(self.corners[1]), Vector(self.corners[2]), Vector(self.corners[6]), Vector(self.corners[5])])
        f5 =  Face([Vector(self.corners[4]), Vector(self.corners[5]), Vector(self.corners[6]), Vector(self.corners[7])])
        f6 =  Face([Vector(self.corners[0]), Vector(self.corners[3]), Vector(self.corners[2]), Vector(self.corners[1])])
        
        poly = [f1, f2 ,f3 ,f4 ,f5, f6]
        if isInPoly(pt, poly) :
            return True
        else:
            return False
    def occupancy_check(self, points):
        for pt in points:
            if self.point_in_cell(pt) :
                self.occupied = True
                break
        return self.occupied

def bbox_cells_center(objects,obj_coords, vol_ = 0.3**3, idx_ = 0 , plot = False) :
    length = vol_**(1/3)
    num_cells, cells_z, cells_y, cells_x = discretize_bbox(objects, length, idx_)
    ncx = cells_x    # number of core mesh cells in x
    ncy = cells_y     # number of core mesh cells in y
    ncz = cells_z     # number of core mesh cells in z
    dx = half_width      # base cell width x
    dy = -half_width     # base cell width y
    dz = half_width     # base cell width z
    hx = dx*np.ones(ncx)
    hy = dy*np.ones(ncy)
    hz = dz*np.ones(ncz)

    if obj_coords[idx_][3][0] > obj_coords[idx_][1][0] :
        coord_origin = 1
    else :
        coord_origin = 3

    x0 = obj_coords[idx_][coord_origin][0]
    y0 = obj_coords[idx_][coord_origin][1]
    z0 = obj_coords[idx_][coord_origin][2]

    #print(objects[idx_].l, objects[idx_].w, objects[idx_].h)
    #print(x0,y0,z0)

    mesh = TensorMesh([hx, hy, hz], x0=[x0, y0,z0])

    hom_bbox = cart2hom(mesh.gridCC)
    bbox_pts = np.dot(rotate(get_rot_angle(obj_coords, num_cells_length = cells_x , idx=idx_, coord_origin = coord_origin),obj_coords[idx_][coord_origin][0],obj_coords[idx_][coord_origin][1]),np.transpose(hom_bbox))
    bbox_pts = np.transpose(bbox_pts)
    bbox_pts = np.array([[x[0]/x[-1]-1, x[1]/x[-1]-1, x[2]] for x in bbox_pts])
    
    if plot:
        mesh.plotGrid()
    
    return bbox_pts

In [ ]:
label_folder = "KITTI/training_labels/label_2/"
calib_folder = "KITTI/data_object_calib/training/calib/"
lidar_folder = "KITTI/data_object_velodyne/training/velodyne/"

index = 5106 #1529 #2081
num = str(index)
num = num.zfill(6)
label_file = label_folder + num + ".txt"
calib_file = calib_folder + num + ".txt"
lidar_file = lidar_folder + num + ".bin"

pcd = get_grd_lvl_point_cloud(lidar_file, height = -1.73 , delta=0.2, display = True)
points = np.asarray(pcd.points)
min_points = np.amin(points, axis =0)
max_points = np.amax(points, axis =0)
print(f"min: {min_points}")
print(f"max: {max_points}")

In [ ]:
ROI_MIN = [4.5,-5]
ROI_MAX = [30, 5]
def create_grid(min_points, max_points, height=-1.73, stride=0.6):
    centers = []
    for x in np.arange(min_points[0], max_points[0], stride):
        for y in np.arange(min_points[1], max_points[1], stride) :
            centers.append([x,y,height])
    return centers

In [ ]:
roi_centers = create_grid(ROI_MIN, ROI_MAX)
tmp1 = create_grid(ROI_MIN, ROI_MAX, -1.63)
tmp2 = create_grid(ROI_MIN, ROI_MAX, - 1.83)
roi = o3d.geometry.PointCloud()
centers = o3d.geometry.PointCloud()
roi.points = o3d.utility.Vector3dVector(np.asarray(tmp1+tmp2))
centers.points = o3d.utility.Vector3dVector(np.asarray(roi_centers))
roi_bbox_aa = roi.get_axis_aligned_bounding_box()
roi_pcd = pcd.crop(roi_bbox_aa)
roi_pcd_down = roi_pcd.voxel_down_sample(0.6)
#o3d.visualization.draw_geometries([roi_pcd_down, centers])

In [ ]:
empty = []
empty_region = []
for pt in roi_centers :
    cell = Cell(pt)
    if not cell.occupancy_check(roi_pcd_down.points) :
        empty.append(cell)
        empty_region.append(cell.center)

empty_region_pcd = o3d.geometry.PointCloud()
empty_region_pcd.points = o3d.utility.Vector3dVector(np.asarray(empty_region))
# o3d.visualization.draw_geometries([empty_region_pcd, pcd])

In [ ]:
# o3d.visualization.draw_geometries([empty_region_pcd, pcd])

In [ ]:
from sklearn.cluster import DBSCAN
from sklearn import metrics
from sklearn.preprocessing import StandardScaler

pts = np.delete(empty_region_pcd.points,-1, axis=1)
db = DBSCAN(eps=0.8, min_samples=5).fit(pts)
core_samples_mask = np.zeros_like(db.labels_, dtype=bool)
core_samples_mask[db.core_sample_indices_] = True
labels = db.labels_

# Number of clusters in labels, ignoring noise if present.
n_clusters_ = len(set(labels)) - (1 if -1 in labels else 0)
n_noise_ = list(labels).count(-1)

print('Total number of points %d' % len(empty_region_pcd.points))
print('Estimated number of clusters: %d' % n_clusters_)
print('Estimated number of noise points: %d' % n_noise_)
if n_clusters_ > 1 :
    print("Silhouette Coefficient: %0.3f"
      % metrics.silhouette_score(pts, labels))

import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams.update({'font.size': 22})
# Black removed and is used for noise instead.
unique_labels = set(labels)
colors = [plt.cm.Spectral(each)
          for each in np.linspace(0, 1, len(unique_labels))]
plt.figure(num=None, figsize=(15, 5), dpi=80, facecolor='w', edgecolor='k')
for k, col in zip(unique_labels, colors):
    if k == -1:
        # Black used for noise.
        col = [0, 0, 0, 1]

    class_member_mask = (labels == k)

    xy = pts[class_member_mask & core_samples_mask]
    plt.plot(xy[:, 0], xy[:, 1], 'o', markerfacecolor=tuple(col),
             markeredgecolor=tuple(col), markersize=5)

    xy = pts[class_member_mask & ~core_samples_mask]
    plt.plot(xy[:, 0], xy[:, 1], 'o', markerfacecolor=tuple(col),
             markeredgecolor=tuple(col), markersize=5)

plt.title('Estimated number of clusters: %d' % n_clusters_)
plt.show()

In [ ]:
def get_clusters(points, clusters):
    n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
    res = []
    for i in range(0, n_clusters):
        tmp = []
        for j in range(0, len(clusters)):
            if clusters[j] == i :
                tmp.append(list(points[j]))
        res.append(tmp)
    return res

# def get_cluster_samples(cluster) :
#     if len(cluster) < 30:
#         return cluster
#     else:
# #         sample_size = round(len(cluster) / 3)
#         if sample_size < 30 :
#             return cluster
#         else:
#             sorted_by_y = sorted(cluster, key = itemgetter(1))
# #             res = random.sample(sorted_by_y, sample_size)
#             res = sorted_by_y[::2]
#             return res

def get_cluster_samples(cluster) :
    if len(cluster) < 30:
        return cluster
    else:
        sorted_by_y = sorted(cluster, key = itemgetter(1))
        res = sorted_by_y[::2]
        print(len(sorted_by_y),len(res))
    return res

In [ ]:
clusters = get_clusters(np.asarray(empty_region_pcd.points), labels)

In [ ]:
def get_angle_between_point_clusters(c1, c2=[[0,0]]) :
    c1_centroid = np.mean(c1, axis = 0).tolist()
    c2_centroid = np.mean(c2, axis = 0).tolist()
    dy = c1_centroid[1] - c2_centroid[1]
    dx = c1_centroid[0] - c2_centroid[0]
    angle_rad = math.atan2(dy,dx)
    return (angle_rad, c1_centroid, c2_centroid)

In [ ]:
def backtrace_shadow_from_pt(centroid, pcd) :
    # doesn't make sense to draw a vector and traverse vector to get cells and find points in cell -- computationally exp
    frustum = {}
    angle_rad = np.arctan(abs(centroid[1])/abs(centroid[0]))
    angle_new = (np.pi/2) - angle_rad
    delta_y = 0.1*np.sin(angle_new)
    delta_x = 0.1*np.cos(angle_new)
    
    points_top =[
            [-0.1, 0, 0],
            [-0.1, 0, -0.1],
            [0.1,0,0],
            [0.1,0,-0.1],
            [centroid[0]-delta_x, centroid[1]-delta_y, centroid[2]+0.1+0.5], # max_grad top
            [centroid[0]-delta_x, centroid[1]-delta_y, centroid[2]+0.1],     
            [centroid[0]+delta_x, centroid[1]+delta_y, centroid[2]+0.1+0.5], #min_grad top
            [centroid[0]+delta_x, centroid[1]+delta_y, centroid[2]+0.1] # min_grad bot
        ]
    
    points_bot =[
            [-0.1, 0, 0],
            [-0.1, 0, -0.1],
            [0.1,0,0],
            [0.1,0,-0.1],
            [centroid[0]+delta_x, centroid[1]+delta_y, centroid[2]+0.1+0.5], #min_grad top
            [centroid[0]+delta_x, centroid[1]+delta_y, centroid[2]+0.1], # min_grad bot
            [centroid[0]-delta_x, centroid[1]-delta_y, centroid[2]+0.1+0.5], # max_grad top
            [centroid[0]-delta_x, centroid[1]-delta_y, centroid[2]+0.1]
        ]
    
    if np.arctan(centroid[1]/centroid[0]) <= 0:
        points = points_top
         # generate faces
        f1 =  Face([Vector(points[0]), Vector(points[2]), Vector(points[3]), Vector(points[1])])
        f2 =  Face([Vector(points[5]), Vector(points[7]), Vector(points[6]), Vector(points[4])])
        f3 =  Face([Vector(points[0]), Vector(points[4]), Vector(points[6]), Vector(points[2])])
        f4 =  Face([Vector(points[7]), Vector(points[5]), Vector(points[1]), Vector(points[3])])
        f5 =  Face([Vector(points[6]), Vector(points[7]), Vector(points[3]), Vector(points[2])])
        f6 =  Face([Vector(points[0]), Vector(points[1]), Vector(points[5]), Vector(points[4])])
    else :
        points = points_bot
         # generate faces
        f1 =  Face([Vector(points[1]), Vector(points[3]), Vector(points[2]), Vector(points[0])])
        f2 =  Face([Vector(points[4]), Vector(points[6]), Vector(points[7]), Vector(points[5])])
        f3 =  Face([Vector(points[2]), Vector(points[6]), Vector(points[4]), Vector(points[0])])
        f4 =  Face([Vector(points[3]), Vector(points[1]), Vector(points[5]), Vector(points[7])])
        f5 =  Face([Vector(points[2]), Vector(points[3]), Vector(points[7]), Vector(points[6])])
        f6 =  Face([Vector(points[4]), Vector(points[5]), Vector(points[1]), Vector(points[0])])
    
    array_ = np.asarray(points)
    vol = o3d.visualization.SelectionPolygonVolume()
    vol.orthogonal_axis = "Y"
    max_ = np.max(array_,axis=0)
    min_ = np.min(array_,axis=0)
    vol.axis_max = max_[1]
    vol.axis_min = min_[1]
    vol.bounding_polygon = o3d.utility.Vector3dVector(array_)
    # crop polgyon from 3D point cloud
    roa_ = vol.crop_point_cloud(pcd)
   
        
    lines_ = [[0,1],[0,2],[0,4], [1,5], [1,3], [2,3], [2,6], [3,7], [4,5], [4,6], [5,7], [6,7]]
    line_set = o3d.geometry.LineSet(points=o3d.utility.Vector3dVector(points), 
                                            lines=o3d.utility.Vector2iVector(lines_))
        
    frustum['frustum_lines'] = line_set
    poly = [f1, f2 ,f3 ,f4 ,f5, f6]
    pts_in_frustum = []
    pt_no = 0
    count = 0
    # count points in faces
    for pt in np.asarray(roa_.points):
        pt_no +=1
        if isInPoly(pt, poly) :
            count +=1
            pts_in_frustum.append(pt)
        else :
            continue
#     pts_in_frustum.append(points)
    frustum['count'] = count
    frustum['pts'] = pts_in_frustum
    
#     frustum['count'] = len(np.asarray(roa_.points))
#     frustum['pts'] = np.asarray(roa_.points)
    return frustum, roa_

def get_gt_bboxes (label_file, calib_file) :
    objs = get_bbox_from_files(label_file, calib_file)
    bboxes = {}
    for obj_id in objs.keys() :
        bbox = o3d.geometry.AxisAlignedBoundingBox()
        bbox = bbox.create_from_points(o3d.utility.Vector3dVector(objs[obj_id]))
        bboxes[obj_id] = bbox
    return bboxes

def shadow_frustum_obj_check(frustum_list, bboxes) :
    res = {}
    idx = 0
    all_ids = []
    for frustum in frustum_list :
        res[idx] = {}
        res[idx]['frustum'] = frustum
        if frustum['count'] == 0 :
            res[idx]['obj_ids'] = None 
        else :
            obj_ids = []
            pts = np.asarray(frustum['pts'])
            for bbox_id in bboxes.keys():
                bbox = bboxes[bbox_id]
                res_ = bbox.get_point_indices_within_bounding_box(o3d.utility.Vector3dVector(pts))
                if not res_ :
                    continue
                else :
                    obj_ids.append(bbox_id)
                    all_ids.append(bbox_id)
            res[idx]['obj_ids'] = obj_ids
        idx +=1
    res['all_objs'] = set(all_ids)
    return res

def get_scene_shadow_res(clusters, pcd, label_file, calib_file) :
    res_dict = {}
    bboxes = get_gt_bboxes (label_file, calib_file)
    for i in range(len(clusters)) :
        frustum_list = []
        samples = get_cluster_samples(clusters[i])
        for pt in samples :
            frus, roa = backtrace_shadow_from_pt(pt, pcd)
            if frus['count'] > 0:
                frustum_list.append(frus)
        res = shadow_frustum_obj_check(frustum_list, bboxes)
        print(f"Cluster : {i} | Obj Match = {res['all_objs']}")
        res_dict[i] = res
    print('--')
    return res_dict

In [ ]:
pcd = get_point_cloud(lidar_file, height = -1.73 , display = False)

In [ ]:
get_scene_shadow_res(clusters, pcd, label_file,calib_file)

In [ ]:
def get_objs_in_roi(objs) :
    obj_list = []
    for obj_id in objs.keys():
        obj_centroid = np.mean(objs[obj_id], axis = 0).tolist()
        print(obj_id, obj_centroid)
        if obj_centroid[0] < 30 and obj_centroid[0] > 4.5:
            if obj_centroid[1] < 5 and obj_centroid[1] > -5 :
                obj_list.append(obj_id)
    return obj_list

In [ ]:
get_objs_in_roi(get_bbox_from_files(label_file, calib_file))

In [ ]:
# get_scene_shadow_res(clusters, pcd, label_file,calib_file)

In [ ]:
frus, roa = backtrace_shadow_from_pt([8.640940967625798, 1.9758680547014367, -0.8988674086316013], pcd)

In [ ]:
frus

In [ ]:
frus['pts'][0]

In [ ]:
pcd2 = o3d.geometry.PointCloud()
pcd2.points = o3d.utility.Vector3dVector(np.asarray(frus['pts']))

In [ ]:
o3d.visualization.draw_geometries([frus['frustum_lines'], pcd2])

In [ ]:
get_bbox_from_files(label_file, calib_file)